# Aerial Image Dataset

This dataset contains 10.000 RGB images, each with resolution 600x600 pixels.

This images has been extracted from _Google Earth_ and comprises 30 classes.

1. airport
2. bare land
3. baseball field
4. beach
5. bridge
6. center
7. church
8. commercial
9. dense residential
10. desert
11. farmland
12. forest
13. industrial
14. meadow
15. medium residential
16. mountain
17. park
18. parking
19. playground
20. pond
21. port
22. railway station
23. resort
24. river
25. school
26. sparse residential
27. square
28. stadium
29. storage tanks
30. viaduct

### __Structure__

```
.
├── README.md
└──  data
     ├── Airport
     │   ├── airport_1.png
     │   ├── airport_2.png
     │   ├── ...
     │   └── airport_360.png
     ├── BareLand
     │   ├── bareland_1.png
     │   ├── ...
     │   └── bareland_310.png
     ├── ...
     └── Viaduct
```

### __Statistics__

+ Total Number of Images: 10,000
+ Image Resolution: 600x600 pixels
+ Scene Categories: 30
+ Dataset Size: 2.6GB

In this notebook we will explore different __regularization techniques__ to prove which configurations are the best for this dataset.

We will develop the following structures of network to prove the perform of the hyperparameters along with the dataset and architecture.

### Experiment 1
+ Start with a small CNN __[S]__
+  Define a medium model __[M]__, this model must be configurable such that we can specify options on whether we want to use batch-norm, dropout, etc.
### Experiment 2: Train with medium model without regularization
+ Train the `MediumModel` without regularization
### Experiment 3: Using meidum model with only
+ We will add 'BatchNorm2d' layers to the feature extractor part of the model.
### Experiment 4: Using Medium Model with only Dropout Regularization
+ Instead of batch norm layers, we will use `Dropout2d` for convolutional layers and `Dropout` for linear layers.
### Experiment 5: Using Medium Model with Batch-Norm and Dropput
+ These cases are slightly better than the ones above
### Experiment 6: Using Medium Model with only Data Augmentation
+ This is a virtual way to increase the number of training data, which helps in model generalization
### Experiment 7: Using a Medium Model with all three techniques
+ The train and validation loss decreases till the last epoch. The gap between these two is very tiny.
### Experiment 8: Using a Large Model with all three techniques

In [1]:
import os
import gc
import random
import math
import warnings
from dataclasses import dataclass
from IPython.display import clear_output
import requests

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.functional import cross_entropy

import torch.optim as optim
import torch.optim.lr_scheduler as lr_scheduler

from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.ops import Conv2dNormActivation


from tqdm import tqdm
from torchinfo import summary
from torchmetrics import MeanMetric
from torchmetrics.classification import MulticlassAccuracy

from matplotlib.ticker import (MultipleLocator, FormatStrFormatter)


# For plotting training and validation metrics.
from livelossplot import PlotLosses
from livelossplot.outputs import MatplotlibPlot, ExtremaPrinter


%matplotlib inline
warnings.filterwarnings(action='ignore', category=UserWarning)

# Text formatting
BOLD = "\033[1m"
END = "\033[0m"

In [3]:
def seed_everything(seed_value):
    np.random.seed(seed_value)
    torch.manual_seed(seed_value)
    torch.cuda.manual_seed_all(seed_value)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [4]:
def get_default_device():
    gpu_available = torch.cuda.is_available()
    return torch.device('cuda' if gpu_available else 'cpu'), gpu_available

In [6]:
# Get dataset
from datasets import load_dataset

ds = load_dataset("blanchon/AID")

c:\Users\sauly\Desktop\Aerial-Image-Classification\ae_venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating train split: 100%|██████████| 10001/10001 [00:00<00:00, 172155.49 examples/s]


In [11]:
def get_mean_std_from_ds(dataset_split, batch_size=64):
    """
    Calcula la media y el std directamente desde un dataset/split de Hugging Face.
    """
    # Transformación para convertir las imágenes de PIL a Tensores normalizados en formato [C, H, W]
    preprocess = transforms.Compose([
        transforms.Resize((224, 224)),  # Redimensionar a tamaño fijo
        transforms.ToTensor()
    ])
    
    total_sum = torch.zeros(3)
    total_sq_sum = torch.zeros(3)
    total_pixels = 0

    for i in range(0, len(dataset_split), batch_size):
        batch_items = dataset_split[i : i + batch_size]
        
        # Extraer las imágenes del batch (suele llamarse 'image' o 'img')
        images_list = batch_items['image']
        
        # Procesar y apilar las imágenes en un solo tensor [batch_size, 3, H, W]
        images = torch.stack([preprocess(img.convert("RGB")) for img in images_list])
        
        batch_size_current = images.size(0)
        pixels_per_batch = batch_size_current * images.size(2) * images.size(3)

        total_sum += images.sum(dim=[0, 2, 3])
        total_sq_sum += (images ** 2).sum(dim=[0, 2, 3])
        total_pixels += pixels_per_batch

    # Calcular media y desviación estándar globales
    mean = total_sum / total_pixels
    var = (total_sq_sum / total_pixels) - (mean ** 2)
    std = torch.sqrt(var)

    mean_tuple = tuple(round(m.item(), 4) for m in mean)
    std_tuple = tuple(round(s.item(), 4) for s in std)

    return mean_tuple, std_tuple

# Para usarlo con tu dataset de Hugging Face (ejecuta esto apuntando al split de entrenamiento):
mean, std = get_mean_std_from_ds(ds['train'])
print(f"Mean: {mean}, Std: {std}")

Mean: (0.3978, 0.4093, 0.3685), Std: (0.2082, 0.1848, 0.1825)


In [13]:
def system_config(SEED_VALUE=42, package_list=None):
    """
    Configures the system environment for PyTorch-based operations.

    Args:
        SEED_VALUE (int): Seed value for random number generation. Default is 42.
        package_list (str): String containing a list of additional packages to install
        for Google Colab or Kaggle. Default is None.

    Returns:
        tuple: A tuple containing the device name as a string and a boolean indicating GPU availability.
    """

    random.seed(SEED_VALUE)
    np.random.seed(SEED_VALUE)
    torch.manual_seed(SEED_VALUE)

    def is_running_in_colab():
        return 'COLAB_GPU' in os.environ

    def is_running_in_kaggle():
        return 'KAGGLE_KERNEL_RUN_TYPE' in os.environ

    #--------------------------------
    # Check for availability of GPUs.
    #--------------------------------
    if torch.cuda.is_available():
        print('Using CUDA GPU')

        # This section for installing packages required by Colab.
        if is_running_in_colab() or is_running_in_kaggle():
            print('Installing required packages...')
            !pip install {package_list}

        # Set the device to the first CUDA device.
        DEVICE = torch.device('cuda')
        print("Device: ", DEVICE)
        GPU_AVAILABLE = True

        torch.cuda.manual_seed(SEED_VALUE)
        torch.cuda.manual_seed_all(SEED_VALUE)

        # Performance and deterministic behavior.
        torch.backends.cudnn.enabled = True       # Provides highly optimized primitives for DL operations.
        torch.backends.cudnn.deterministic = True # Insures deterministic even when above cudnn is enabled.
        torch.backends.cudnn.benchmark = False    # Setting to True can cause non-deterministic behavior.

    elif torch.backends.mps.is_available() and torch.backends.mps.is_built():
        print('Using Apple Silicon GPU')

        # Set the device to the Apple Silicon GPU Metal Performance Shader (MPS).
        DEVICE = torch.device("mps")
        print("Device: ", DEVICE)
        # Environment variable that allows PyTorch to fall back to CPU execution
        # when encountering operations that are not currently supported by MPS.
        os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'
        GPU_AVAILABLE = True

        torch.mps.manual_seed(SEED_VALUE)
        torch.use_deterministic_algorithms(True)

    else:
        print('Using CPU')
        DEVICE = torch.device('cpu')
        print("Device: ", DEVICE)
        GPU_AVAILABLE = False

        if is_running_in_colab() or is_running_in_kaggle():
            print('Installing required packages...')
            !pip install {package_list}
            print('Note: Change runtime type to GPU for better performance.')

        torch.use_deterministic_algorithms(True)

    return str(DEVICE), GPU_AVAILABLE

In [14]:
DEVICE, GPU_AVAILABLE = system_config()

Using CPU
Device:  cpu


### Transform the model from Hugging Face format to ImageFolder

In [16]:
def save_hf_dataset_to_imagefolder(dataset_split, output_dir="./dataset_export"):
    # Crear estructura de carpetas para ImageFolder
    train_dir = os.path.join(output_dir, "training", "training")
    os.makedirs(train_dir, exist_ok=True)
    
    for idx, item in enumerate(dataset_split):
        image = item['image']   # Objeto PIL Image
        label = item['label']   # ID de la clase (número)
        
        # Obtener el nombre de la clase si está disponible, o usar el número
        class_name = str(label)
        class_dir = os.path.join(train_dir, class_name)
        os.makedirs(class_dir, exist_ok=True)
        
        # Guardar imagen como JPG
        img_path = os.path.join(class_dir, f"{idx}.jpg")
        if isinstance(image, Image.Image):
            image.convert("RGB").save(img_path)

# Uso:
# save_hf_dataset_to_imagefolder(ds['train'])
# Luego ya puedes usar ImageFolder apuntando a './dataset_export'

In [18]:
print(ds['train'].features)

{'image': Image(mode=None, decode=True)}


In [19]:
import torch
from torch.utils.data import DataLoader
from torchvision import transforms

def get_mean_std_from_hf(dataset_split, batch_size=64):
    """
    Calcula la media y el std directamente de un dataset de Hugging Face 
    solo usando las imágenes, sin importar las etiquetas.
    """
    preprocess = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor()
    ])
    
    total_sum = torch.zeros(3)
    total_sq_sum = torch.zeros(3)
    total_pixels = 0

    # Iterar por lotes sobre el dataset de Hugging Face
    for i in range(0, len(dataset_split), batch_size):
        batch_items = dataset_split[i : i + batch_size]
        images_list = batch_items['image']
        
        # Procesar, asegurar formato RGB [3, H, W] y apilar
        images = torch.stack([preprocess(img.convert("RGB")) for img in images_list])
        
        batch_size_current = images.size(0)
        pixels_per_batch = batch_size_current * images.size(2) * images.size(3)

        total_sum += images.sum(dim=[0, 2, 3])
        total_sq_sum += (images ** 2).sum(dim=[0, 2, 3])
        total_pixels += pixels_per_batch

    # Calcular media y desviación estándar globales por canal (RGB)
    mean = total_sum / total_pixels
    var = (total_sq_sum / total_pixels) - (mean ** 2)
    std = torch.sqrt(var)

    mean_tuple = tuple(round(m.item(), 4) for m in mean)
    std_tuple = tuple(round(s.item(), 4) for s in std)

    return mean_tuple, std_tuple

# ===
mean, std = get_mean_std_from_hf(ds['train'])

print(f"Mean: {mean}")
print(f"Std: {std}")

Mean: (0.3978, 0.4093, 0.3685)
Std: (0.2082, 0.1848, 0.1825)


## 1 Training and Dataset Configurations

In [20]:
@dataclass(frozen=True)
class DatasetConfig:
    HEIGHT: int = 600
    WIDTH: int = 600
    CHANNELS: int = 3
    NUM_CLASSES: int = 32
    MEAN: tuple = mean
    STD: tuple = std

@dataclass(frozen=True)
class TrainingConfig:
    BATCH_SIZE: int = 32
    NUM_EPOCHS: int = 60
    LEARNING_RATE: float = 0.1
    DECAY_RATE: float = 0.1
    WARMUP_EPOCHS: int = 10
    NUM_WORKERS: int = os.cpu_count()